# Demo: Analisi emotiva di un singolo commento con ELIta

Questo notebook mostra **passo per passo** come il metodo finale assegna un'emozione a un commento del corpus `r/Italia — notizie`.

**Metodo finale**: lessico ELIta ibrido (α=0.5) + ALL_STOPWORDS + soglia di distintività ≥ 0.06

Per ogni commento selezionato:
1. Si mostrano i token lemmatizzati e quali vengono usati
2. Si mostra il contributo emotivo di ogni parola trovata in ELIta
3. Si calcola il vettore emotivo aggregato e l'emozione dominante

## Setup e caricamento dati

In [1]:
import pandas as pd
import numpy as np
import emoji
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
from IPython.display import display

CORPUS_CSV   = Path('corpus_Italia_notizie.csv')
TOKENS_CSV   = Path('tokens_Italia_notizie.csv')
ELITA_CSV    = Path('../Fase1/ELIta_INTENSITY_Matrix.csv')
ALPHA_02_CSV = Path('../Fase2/output_csv/elita_recalculated_0_2.csv')
ALPHA_05_CSV = Path('../Fase2/output_csv/elita_recalculated_0_5.csv')
ALPHA_08_CSV = Path('../Fase2/output_csv/elita_recalculated_0_8.csv')

BASIC_EMOTIONS = ['gioia','tristezza','rabbia','paura','disgusto','fiducia','sorpresa','aspettativa']
EMOTION_COLORS = {
    'gioia':'#FDD835', 'tristezza':'#1E88E5', 'rabbia':'#E53935', 'paura':'#43A047',
    'disgusto':'#8E24AA', 'fiducia':'#81C784', 'sorpresa':'#039BE5',
    'aspettativa':'#FB8C00', 'neutrale':'#9E9E9E',
}

EMOTIONAL_STOPWORDS = {
    'avere','essere','fare','stare','dare','andare','venire',
    'potere','volere','dovere','sapere','vedere','sentire',
    'trovare','pensare','dire','parlare','guardare','tenere',
    'portare','prendere','mettere','lasciare','passare','uscire',
    'entrare','tornare','rimanere','iniziare','finire','continuare',
    'cominciare','provare','riuscire','sembrare','diventare',
    'cosa','modo','parte','punto','volta','anno','tempo','caso',
    'fatto','posto','tipo','gente','persona','vita','mondo',
    'uomo','donna','bambino','figlio','figlia','padre','madre',
    'altro','solo','grande','piccolo','nuovo','vecchio','primo',
    'ultimo','stesso','proprio','bello','buono','lungo','alto',
    'più','bene','male','molto','poco','tanto','tutto','niente',
}
TOPIC_STOPWORDS = {
    'notizia','notizie','giornale','giornali','giornalista','giornalismo',
    'informazione','informazioni','articolo','articoli','media','fonte',
    'fonti','testata','redazione','titolo','telegiornale',
}
ALL_STOPWORDS = EMOTIONAL_STOPWORDS | TOPIC_STOPWORDS
DIST_THRESHOLD = 0.06
POS_FILTER     = {'ADJ', 'NOUN', 'VERB'}

print('Configurazione caricata.')

Configurazione caricata.


In [2]:
df_corpus = pd.read_csv(CORPUS_CSV)
df_tokens = pd.read_csv(TOKENS_CSV)
df_tokens['lemma'] = df_tokens['lemma'].astype(str).str.lower().str.strip()
df_tokens['pos']   = df_tokens['pos'].astype(str).str.upper().str.strip()

def is_not_emoji(text):
    return emoji.emoji_count(str(text)) == 0

# Metodo finale: solo α=0.5
df_elita_final = pd.read_csv(ALPHA_05_CSV, index_col=0)
df_elita_final.index = df_elita_final.index.astype(str).str.lower().str.strip()
df_elita_final = df_elita_final[BASIC_EMOTIONS].fillna(0)

# Pre-calcolo distintività sul lessico originale (stessa soglia di Confronto_notizie.ipynb)
df_elita_orig = pd.read_csv(ELITA_CSV, index_col=0)
df_elita_orig = df_elita_orig[df_elita_orig.index.map(is_not_emoji)]
df_elita_orig.index = df_elita_orig.index.astype(str).str.lower().str.strip()
df_elita_orig = df_elita_orig[BASIC_EMOTIONS].fillna(0)

def calc_dist(df_e):
    def _row(row):
        sv = sorted(row.values, reverse=True)
        max1, max2 = sv[0], sv[1]
        mn = np.mean(row.values)
        if max1 == 0:
            return 0.0
        return ((max1 - max2) / (max1 + 1e-9)) * (max1 - mn)
    return df_e[BASIC_EMOTIONS].apply(_row, axis=1)

dist_orig = calc_dist(df_elita_orig)
ELITA_IDX_FILTERED = set(dist_orig[dist_orig >= DIST_THRESHOLD].index)

print(f'Corpus: {len(df_corpus)} commenti | Token: {len(df_tokens)}')
print(f'Parole ELIta (α=0.5) con dist ≥ {DIST_THRESHOLD}: {len(ELITA_IDX_FILTERED)}')

Corpus: 700 commenti | Token: 64812
Parole ELIta (α=0.5) con dist ≥ 0.06: 3107


## Selezione del commento

Modifica `COMMENT_INDEX` (0–699) per scegliere un commento diverso, oppure imposta `COMMENT_ID` con un ID specifico.

In [3]:
COMMENT_INDEX = 16      # <--- cambia qui (0-699)
COMMENT_ID    = None   # oppure specifica un ID, es. 'kfr4gvl'

if COMMENT_ID:
    row = df_corpus[df_corpus['comment_id'] == COMMENT_ID].iloc[0]
else:
    row = df_corpus.iloc[COMMENT_INDEX]

CID  = row['comment_id']
TEXT = row['comment_text']

print(f'ID commento : {CID}')
print(f'Autore      : {row["comment_author"]}')
print(f'Score Reddit: {row["comment_score"]}')
print()
print('Testo:')
print('-' * 70)
print(TEXT)
print('-' * 70)

ID commento : kf8t1mk
Autore      : solitaSE
Score Reddit: 1

Testo:
----------------------------------------------------------------------
Guarda pure io ho una memoria pessima ma quello che mi aiuta è leggere tanto le notizie (basta seguire un paio di giornali nazionali e internazionali sui social per essere aggiornati su quello che succede nel mondo), e sopratutto essere curiosi e farsi domande…per esempio sto guardando un film sulla prima guerra mondiale, metto pausa e vado a leggere su wiki di che periodo stiamo parlando, finito il film faccio ricerca su personaggi specifici e approfondimenti sulla storia etc…mi ricorderò tutta questa info da qui a un paio di mesi? Probabilmente no ☺️ ma a forza di farmi domande e continuare a informarmi sicuramente qualcosa rimarrà dentro
----------------------------------------------------------------------


## Token e lemmi del commento

In [4]:
df_tok = df_tokens[df_tokens['comment_id'] == CID].copy()

elita_idx_all = set(df_elita_final.index)

df_tok['in_ELIta']  = df_tok['lemma'].isin(elita_idx_all)
df_tok['pos_ok']    = df_tok['pos'].isin(POS_FILTER)
df_tok['stopword']  = df_tok['lemma'].isin(ALL_STOPWORDS)
df_tok['dist_ok']   = df_tok['lemma'].isin(ELITA_IDX_FILTERED)
df_tok['usato']     = df_tok['in_ELIta'] & df_tok['pos_ok'] & ~df_tok['stopword'] & df_tok['dist_ok']

print(f'Token totali nel commento      : {len(df_tok)}')
print(f'Con POS valida (ADJ/NOUN/VERB) : {df_tok["pos_ok"].sum()}')
print(f'Trovati in ELIta               : {df_tok["in_ELIta"].sum()}')
print(f'Stopwords rimosse              : {(df_tok["in_ELIta"] & df_tok["stopword"]).sum()}')
print(f'Sotto soglia distintività      : {(df_tok["in_ELIta"] & df_tok["pos_ok"] & ~df_tok["stopword"] & ~df_tok["dist_ok"]).sum()}')
print(f'Token usati per l\'analisi      : {df_tok["usato"].sum()}')
print()

display(df_tok[['token','lemma','pos','in_ELIta','stopword','dist_ok','usato']].reset_index(drop=True))

Token totali nel commento      : 110
Con POS valida (ADJ/NOUN/VERB) : 53
Trovati in ELIta               : 46
Stopwords rimosse              : 13
Sotto soglia distintività      : 17
Token usati per l'analisi      : 13



,token,lemma,pos,in_ELIta,stopword,dist_ok,usato
0,Guarda,guarda,VERB,False,False,False,False
1,pure,pure,ADV,False,False,False,False
2,io,io,PRON,False,False,False,False
3,ho,avere,VERB,True,True,False,False
4,una,uno,DET,False,False,False,False
...,...,...,...,...,...,...,...
105,informarmi,informare mi,NOUN,False,False,False,False
106,sicuramente,sicuramente,ADV,True,False,False,False
107,qualcosa,qualcosa,PRON,False,False,False,False
108,rimarrà,rimarrà,VERB,False,False,False,False


## Contributo emotivo per parola

Per ogni lemma usato nell'analisi, mostriamo il vettore emotivo da ELIta (versione originale).

In [5]:
lemmi_usati = df_tok[df_tok['usato']]['lemma'].tolist()

if not lemmi_usati:
    print('Nessun lemma utile trovato in questo commento.')
else:
    contrib_rows = []
    for lemma in lemmi_usati:
        scores = df_elita_final.loc[lemma, BASIC_EMOTIONS].to_dict()
        dom    = max(scores, key=scores.get)
        scores['lemma']   = lemma
        scores['dom_emo'] = dom
        contrib_rows.append(scores)

    df_contrib = pd.DataFrame(contrib_rows)
    cols_show  = ['lemma'] + BASIC_EMOTIONS + ['dom_emo']

    totals  = df_contrib[BASIC_EMOTIONS].sum()
    dom_tot = totals.idxmax()

    print('Contributi emotivi per lemma (ELIta α=0.5):')
    display(
        df_contrib[cols_show]
        .style
        .background_gradient(subset=BASIC_EMOTIONS, cmap='YlOrRd', axis=None)
        .format({e: '{:.2f}' for e in BASIC_EMOTIONS})
    )
    print()
    print('TOTALE per emozione:')
    print(totals.round(3).to_string())
    print(f'\n=> Emozione dominante: {dom_tot.upper()}')

Contributi emotivi per lemma (ELIta α=0.5):


,lemma,gioia,tristezza,rabbia,paura,disgusto,fiducia,sorpresa,aspettativa,dom_emo
0,leggere,0.85,0.30,0.14,0.14,0.18,0.68,0.50,0.64,gioia
1,aggiornare,0.67,0.18,0.19,0.29,0.09,0.62,0.60,0.78,aspettativa
2,curioso,0.59,0.12,0.27,0.34,0.11,0.51,0.82,0.69,sorpresa
3,domanda,0.43,0.21,0.22,0.53,0.18,0.62,0.56,0.93,aspettativa
4,film,0.66,0.53,0.44,0.56,0.28,0.27,0.62,0.54,gioia
5,pausa,0.54,0.21,0.15,0.31,0.09,0.52,0.30,0.71,aspettativa
6,leggere,0.85,0.30,0.14,0.14,0.18,0.68,0.50,0.64,gioia
7,periodo,0.29,0.25,0.26,0.38,0.18,0.44,0.50,0.68,aspettativa
8,film,0.66,0.53,0.44,0.56,0.28,0.27,0.62,0.54,gioia
9,ricerca,0.35,0.21,0.21,0.38,0.15,0.59,0.47,0.84,aspettativa



TOTALE per emozione:
gioia          7.389
tristezza      3.722
rabbia         3.172
paura          4.749
disgusto       2.163
fiducia        6.814
sorpresa       6.892
aspettativa    9.039

=> Emozione dominante: ASPETTATIVA


## Profilo emotivo aggregato — radar chart

In [6]:
if not lemmi_usati:
    print('Nessun lemma utile trovato.')
else:
    emos_loop = BASIC_EMOTIONS + [BASIC_EMOTIONS[0]]
    vals = [totals[e] for e in emos_loop]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=vals, theta=emos_loop,
        fill='toself', opacity=0.6,
        name='ELIta α=0.5',
        line_color='#FB8C00'
    ))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True)),
        title=f'Profilo emotivo — commento {CID} (metodo finale)',
        height=480
    )
    fig.show()

## Barchart: score per emozione

In [7]:
if not lemmi_usati:
    print('Nessun lemma utile trovato.')
else:
    bar_colors = [EMOTION_COLORS[e] for e in BASIC_EMOTIONS]
    fig = go.Figure(go.Bar(
        x=BASIC_EMOTIONS,
        y=[totals[e] for e in BASIC_EMOTIONS],
        marker_color=bar_colors,
        text=[f'{totals[e]:.2f}' for e in BASIC_EMOTIONS],
        textposition='outside'
    ))
    fig.update_layout(
        title=f'Score emotivi aggregati — commento {CID} (ELIta α=0.5)',
        xaxis_title='Emozione',
        yaxis_title='Score aggregato',
        height=430
    )
    fig.show()

    print(f'\nLemmi usati ({len(lemmi_usati)}): {lemmi_usati}')
    print(f'Emozione dominante: {dom_tot.upper()} (score: {totals[dom_tot]:.3f})')


Lemmi usati (13): ['leggere', 'aggiornare', 'curioso', 'domanda', 'film', 'pausa', 'leggere', 'periodo', 'film', 'ricerca', 'mese', 'forza', 'domanda']
Emozione dominante: ASPETTATIVA (score: 9.039)


## Analisi con modello Ekman (6 emozioni)

Ripetiamo l'analisi escludendo **fiducia** e **aspettativa** (emozioni cognitive/valutative
presenti in Plutchik ma non in Ekman).

La soglia di distintività viene ricalcolata sulle sole 6 colonne Ekman:
la media cambia (6 valori invece di 8), quindi cambiano anche i valori di `d`
e il sottoinsieme di parole che supera la soglia.

In [8]:
EKMAN_EMOTIONS = ['gioia','tristezza','rabbia','paura','disgusto','sorpresa']

# Distintività calcolata sulle sole 6 colonne Ekman
def calc_dist_ekman(df_e):
    def _row(row):
        sv = sorted(row.values, reverse=True)
        max1, max2 = sv[0], sv[1]
        mn = np.mean(row.values)
        if max1 == 0:
            return 0.0
        return ((max1 - max2) / (max1 + 1e-9)) * (max1 - mn)
    return df_e[EKMAN_EMOTIONS].apply(_row, axis=1)

dist_ekman = calc_dist_ekman(df_elita_orig)
ELITA_IDX_EKMAN = set(dist_ekman[dist_ekman >= DIST_THRESHOLD].index)

# Ricalcolo dei token usati con filtro Ekman
df_tok['dist_ok_ekman'] = df_tok['lemma'].isin(ELITA_IDX_EKMAN)
df_tok['usato_ekman']   = (
    df_tok['in_ELIta'] & df_tok['pos_ok'] &
    ~df_tok['stopword'] & df_tok['dist_ok_ekman']
)

lemmi_ekman = df_tok[df_tok['usato_ekman']]['lemma'].tolist()

print('Token usati — Plutchik (8): {:d}  |  Ekman (6): {:d}'.format(
    df_tok['usato'].sum(), df_tok['usato_ekman'].sum()))
print()

if not lemmi_ekman:
    print('Nessun lemma utile trovato con il modello Ekman.')
else:
    contrib_ekman = []
    for lemma in lemmi_ekman:
        scores = df_elita_final.loc[lemma, EKMAN_EMOTIONS].to_dict()
        dom    = max(scores, key=scores.get)
        scores['lemma']   = lemma
        scores['dom_emo'] = dom
        contrib_ekman.append(scores)

    df_contrib_ekman = pd.DataFrame(contrib_ekman)
    totals_ekman     = df_contrib_ekman[EKMAN_EMOTIONS].sum()
    dom_ekman        = totals_ekman.idxmax()

    print('Contributi emotivi per lemma (Ekman, ELIta α=0.5):')
    display(
        df_contrib_ekman[['lemma'] + EKMAN_EMOTIONS + ['dom_emo']]
        .style
        .background_gradient(subset=EKMAN_EMOTIONS, cmap='YlOrRd', axis=None)
        .format({e: '{:.2f}' for e in EKMAN_EMOTIONS})
    )
    print()
    print('TOTALE per emozione (Ekman):')
    print(totals_ekman.round(3).to_string())
    print('\n=> Emozione dominante Ekman: {}'.format(dom_ekman.upper()))

Token usati — Plutchik (8): 13  |  Ekman (6): 13

Contributi emotivi per lemma (Ekman, ELIta α=0.5):


,lemma,gioia,tristezza,rabbia,paura,disgusto,sorpresa,dom_emo
0,aiutare,0.83,0.31,0.12,0.19,0.07,0.47,gioia
1,leggere,0.85,0.30,0.14,0.14,0.18,0.50,gioia
2,internazionale,0.58,0.08,0.07,0.14,0.05,0.38,gioia
3,aggiornare,0.67,0.18,0.19,0.29,0.09,0.60,gioia
4,succedere,0.40,0.30,0.21,0.55,0.13,0.74,sorpresa
5,curioso,0.59,0.12,0.27,0.34,0.11,0.82,sorpresa
6,esempio,0.39,0.14,0.15,0.15,0.12,0.54,sorpresa
7,pausa,0.54,0.21,0.15,0.31,0.09,0.30,gioia
8,leggere,0.85,0.30,0.14,0.14,0.18,0.50,gioia
9,personaggio,0.26,0.21,0.23,0.31,0.19,0.56,sorpresa



TOTALE per emozione (Ekman):
gioia        7.553
tristezza    3.059
rabbia       2.346
paura        3.289
disgusto     1.585
sorpresa     6.579

=> Emozione dominante Ekman: GIOIA


In [9]:
if lemmi_ekman:
    # Radar Ekman
    emos_loop_ek = EKMAN_EMOTIONS + [EKMAN_EMOTIONS[0]]
    vals_ek = [totals_ekman[e] for e in emos_loop_ek]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=vals_ek, theta=emos_loop_ek,
        fill='toself', opacity=0.6,
        name='Ekman (6)', line_color='#1E88E5'
    ))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True)),
        title='Profilo emotivo Ekman — commento {} (α=0.5)'.format(CID),
        height=480
    )
    fig.show()

In [10]:
if lemmi_ekman:
    # Confronto Plutchik vs Ekman sulle 6 emozioni in comune
    emos_comuni = EKMAN_EMOTIONS

    fig = make_subplots(rows=1, cols=2,
        subplot_titles=['Plutchik (8 emozioni)', 'Ekman (6 emozioni)'],
        horizontal_spacing=0.12)

    bar_colors = [EMOTION_COLORS[e] for e in emos_comuni]

    fig.add_trace(go.Bar(
        x=emos_comuni,
        y=[totals[e] for e in emos_comuni],
        marker_color=bar_colors,
        text=['{:.2f}'.format(totals[e]) for e in emos_comuni],
        textposition='outside', name='Plutchik'),
        row=1, col=1)

    fig.add_trace(go.Bar(
        x=emos_comuni,
        y=[totals_ekman[e] for e in emos_comuni],
        marker_color=bar_colors,
        text=['{:.2f}'.format(totals_ekman[e]) for e in emos_comuni],
        textposition='outside', name='Ekman'),
        row=1, col=2)

    fig.update_layout(
        title='Confronto Plutchik vs Ekman — commento {} (6 emozioni in comune)'.format(CID),
        showlegend=False, height=430)
    fig.show()

    print('Emozione dominante — Plutchik: {:s}  |  Ekman: {:s}'.format(
        dom_tot.upper(), dom_ekman.upper()))
    if dom_tot == dom_ekman:
        print('=> I due modelli concordano.')
    elif dom_tot in ['aspettativa','fiducia']:
        print('=> In Plutchik dominava {} (esclusa da Ekman).'.format(dom_tot))
        print('   In Ekman emerge {} come emozione principale.'.format(dom_ekman))
    else:
        print('=> I due modelli assegnano emozioni diverse.')

Emozione dominante — Plutchik: ASPETTATIVA  |  Ekman: GIOIA
=> In Plutchik dominava aspettativa (esclusa da Ekman).
   In Ekman emerge gioia come emozione principale.
